<div style="padding: 20px; background: linear-gradient(90deg, #8E2DE2 0%, #4A00E0 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">📏 Module 9.1: RAG Evaluation with RAGAS</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Automated, reference-free evaluation for RAG pipelines.</p>
</div>

---

## 1. Why Evaluate RAG?

Evaluating LLMs and RAG pipelines is notoriously difficult. Unlike traditional machine learning (where you measure Accuracy, F1 Score, etc.), RAG outputs natural language. 

**RAGAS** (Retrieval Augmented Generation Assessment) provides automated metrics to evaluate your pipeline using an LLM-as-a-judge approach.

## 2. Key Metrics

| Metric | Component Measured | Description | Target Range |
| :--- | :--- | :--- | :--- |
| **Faithfulness** | Generator (LLM) | Does the answer hallucinate? Is everything stated strictly grounded in the retrieved context? | `0 → 1` |
| **Answer Relevancy** | Generator (LLM) | Does the generated answer actually address the user's question? | `0 → 1` |
| **Context Precision** | Retriever | Are the retrieved chunks relevant? (High precision = fewer useless chunks) | `0 → 1` |
| **Context Recall** | Retriever | Did we retrieve all the necessary facts to answer the question? | `0 → 1` |

In this notebook, we will use a **Judge LLM** (`llama-3.1-8b-instant`) to evaluate a mock dataset.

### Course alignment and free-first stack

- Covers: RAGAS metrics, evaluation datasets, LangChain wrappers, and aggregate reporting.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

print("Libraries loaded successfully.")

## 3. Building an Evaluation Dataset

To evaluate a RAG pipeline, you need a dataset. In RAGAS, a typical `SingleTurnSample` requires:
1. `user_input` (The question)
2. `response` (The answer generated by your RAG pipeline)
3. `retrieved_contexts` (The chunks retrieved from the DB)
4. `reference` (Optional: The ground truth answer if available)

Let's create 3 sample rows. Notice how **Sample 2** contains a deliberate hallucination!

In [ ]:
samples = [
    # Sample 1: Perfect retrieval and generation
    SingleTurnSample(
        user_input="What is RAG?",
        response="RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a generative model.",
        retrieved_contexts=[
            "RAG (Retrieval-Augmented Generation) is a technique that retrieves relevant documents and passes them to an LLM.",
            "RAG was designed to reduce hallucination and provide up-to-date knowledge.",
        ],
        reference="RAG is a technique that retrieves relevant documents and uses them to ground LLM generation.",
    ),
    
    # Sample 2: Deliberate Hallucination (Context is correct, but LLM ignores it)
    SingleTurnSample(
        user_input="What is the capital of France?",
        response="The capital of France is Berlin.",  # DELIBERATE HALLUCINATION
        retrieved_contexts=[
            "Paris is the capital and largest city of France.",
            "France is a country in Western Europe.",
        ],
        reference="Paris is the capital of France.",
    ),
    
    # Sample 3: Solid generation, mediocre retrieval
    SingleTurnSample(
        user_input="How does vector similarity search work?",
        response="Vector similarity search computes the cosine similarity between the query embedding and stored embeddings to find the most relevant documents.",
        retrieved_contexts=[
            "Vector stores index embeddings and use distance metrics like cosine similarity to find similar vectors.",
            "FAISS and Chroma are popular vector stores used for similarity search.",
        ],
        reference="Vector similarity search embeds the query and computes distances to stored vectors to retrieve the nearest neighbours.",
    ),
]

dataset = EvaluationDataset(samples=samples)
print("Dataset constructed with 3 samples.")

## 4. Running the RAGAS Evaluator

RAGAS uses an LLM to "judge" the quality of the outputs. 
We will initialize `ChatGroq` as the judge LLM and `HuggingFaceEmbeddings` for the embedding evaluations.

In [ ]:
import os
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    # 1. Initialize Models
    # RAGAS requires wrappers for Langchain models in v0.3.x
    llm = LangchainLLMWrapper(ChatGroq(model="llama-3.1-8b-instant", temperature=0))
    embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")))
    
    print("\n--- Running RAGAS Evaluation ---")
    # 2. Run Evaluate (This will make API calls to Groq)
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=llm,
        embeddings=embeddings,
    )
    
    print("\n✅ Evaluation Complete!\n")
    print("="*60)
    
    # Convert results to pandas for beautiful display
    df = results.to_pandas()
    display(df[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]])
    
    print("\n📊 Mean Aggregate Scores:")
    for col in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        print(f"  • {col.title():25}: {df[col].mean():.4f}")
else:
    print("GROQ_API_KEY missing. Please add it to your .env file.")

### Interpreting Results
- Notice **Sample 2** (The Capital of France). Because we deliberately answered "Berlin" while the retrieved context correctly said "Paris", the `faithfulness` score should be very low (close to 0.0). The LLM hallucinated, and the Judge LLM correctly penalized it!
- **Sample 1** and **Sample 3** should have high scores across the board since the responses are grounded in the context.

---